In [43]:
#import libraries
import os
import sys
import numpy as np
import pandas as pd


In [44]:
#open remoteness index dataset
remoteness = pd.read_csv(os.path.join('..','..','data','raw','2021_Remoteness_Index.csv'))

# Display the first few rows of each dataset
print("Remoteness Index dataset:")
print(remoteness.head())


Remoteness Index dataset:
   Unnamed: 0  Pruid   CSDuid                   CSDname CSDtype  \
0           0     10  1001101  Division No.  1, Subd. V     SNO   
1           1     10  1001105       Portugal Cove South       T   
2           2     10  1001113                 Trepassey       T   
3           3     10  1001120               St. Shott's       T   
4           4     10  1001124  Division No.  1, Subd. U     SNO   

   Index_of_remoteness  CSDpop2021             DGUID  
0             0.380336        55.0  2021A00051001101  
1             0.391640        86.0  2021A00051001105  
2             0.397290       405.0  2021A00051001113  
3             0.403400        55.0  2021A00051001120  
4             0.349645      1373.0  2021A00051001124  


In [45]:
remoteness["CSDuid"]

0       1001101
1       1001105
2       1001113
3       1001120
4       1001124
         ...   
5156    6208068
5157    6208073
5158    6208081
5159    6208087
5160    6208098
Name: CSDuid, Length: 5161, dtype: int64

In [46]:
#replace the csd name "north vancouver" that has a csd code 5915046 in column one, to "north vancouver district municipality"
#replace the csd name "north vancouver" that has a csd code 5915051 in column one, to "north vancouver city"

# Replace North Vancouver entries based on CSD codes
remoteness.loc[remoteness['CSDuid'] == 5915046, 'CSDname'] = 'North Vancouver District'
remoteness.loc[remoteness['CSDuid'] == 5915051, 'CSDname'] = 'North Vancouver City'

#rename sun peaks mountain to sun peaks mountain resort in the remoteness data
remoteness['CSDname'] = remoteness['CSDname'].replace('Sun Peaks Mountain', 'Sun Peaks Mountain Resort')

#replace the csd name "langley" that has a csd code 5915001 in column one, to "langley township"
remoteness.loc[remoteness['CSDuid'] == 5915001, 'CSDname'] = 'Langley Township'

#replace the csd name "langley" that has a csd code 5915002 in column one, to "langley city"
remoteness.loc[remoteness['CSDuid'] == 5915002, 'CSDname'] = 'Langley City'

# Replace Sechelt (Part) entries based on CSD codes
remoteness.loc[remoteness['CSDuid'] == 5929803, 'CSDname'] = 'Sechelt IGD'


In [47]:
remoteness.loc[remoteness['CSDuid'] == 5915046, 'CSDname']

4462    North Vancouver District
Name: CSDname, dtype: object

In [48]:
#remove cities with the same name in the remoteness dataset that are not from BC

# Define the correct BC CSD codes
bc_csd_codes = {
    'Victoria': 5917034,
    'Richmond': 5915015,
    'Armstrong': 5937028,
    'Nelson': 5903015,
    'Kent': 5909032,
    'Hope': 5909009,
    'Esquimalt': 5917040,
    'Alert Bay': 5943008,
    'Langford' : 5917044,
}

# Remove duplicates by keeping only the BC CSD codes
for city_name, correct_code in bc_csd_codes.items():
    # Remove all rows with this city name that don't have the correct code
    remoteness = remoteness[~((remoteness['CSDname'] == city_name) & 
                          (remoteness['CSDuid'] != correct_code))]

print("Removed duplicate cities from 2021 dataset")


Removed duplicate cities from 2021 dataset


In [49]:
print(remoteness)

      Unnamed: 0  Pruid   CSDuid                   CSDname CSDtype  \
0              0     10  1001101  Division No.  1, Subd. V     SNO   
1              1     10  1001105       Portugal Cove South       T   
2              2     10  1001113                 Trepassey       T   
3              3     10  1001120               St. Shott's       T   
4              4     10  1001124  Division No.  1, Subd. U     SNO   
...          ...    ...      ...                       ...     ...   
5156        5156     62  6208068               Umingmaktok     SET   
5157        5157     62  6208073             Cambridge Bay     HAM   
5158        5158     62  6208081                Gjoa Haven     HAM   
5159        5159     62  6208087                  Taloyoak     HAM   
5160        5160     62  6208098    Kitikmeot, Unorganized      NO   

      Index_of_remoteness  CSDpop2021             DGUID  
0                0.380336        55.0  2021A00051001101  
1                0.391640        86.0  2021

In [50]:
# Get community names used in the wellbeing dataset to match with remoteness dataset
wellbeing_communities = pd.read_csv(os.path.join('..','..','data','processed','matched_cwb_2021.csv'), usecols=['CSD Name 2021'])

In [51]:
print(len(wellbeing_communities['CSD Name 2021'].unique()))

145


In [52]:
# Get unique community names from each dataset as a list
remoteness_communities = set(remoteness['CSDname'].dropna())

# Find exact matches (as a list)
exact_matches_2021 = remoteness_communities.intersection(set(wellbeing_communities['CSD Name 2021']))

print(f"Exact matches with 2021 wellbeing: {len(exact_matches_2021)}")

# Get unmatched communities by subtracting lists
unmatched_communities = set(wellbeing_communities['CSD Name 2021']) - exact_matches_2021

print("\nAll unmatched communities:")
for community in sorted(list(unmatched_communities)):
    print(f"  {community}")


Exact matches with 2021 wellbeing: 145

All unmatched communities:


In [53]:
remoteness_subset = remoteness[remoteness['CSDname'].isin(exact_matches_2021)].copy()

In [54]:
remoteness_subset.head()

,Unnamed: 0,Pruid,CSDuid,CSDname,CSDtype,Index_of_remoteness,CSDpop2021,DGUID
4304,4304,59,5901006,Sparwood,DM,0.387226,4148.0,2021A00055901006
4305,4305,59,5901012,Fernie,CY,0.427617,6320.0,2021A00055901012
4308,4308,59,5901022,Cranbrook,CY,0.385609,20499.0,2021A00055901022
4309,4309,59,5901028,Kimberley,CY,0.409831,8115.0,2021A00055901028
4312,4312,59,5901039,Invermere,DM,0.460123,3917.0,2021A00055901039


In [55]:
# Create tidy version of remoteness data

tidy_remoteness = pd.DataFrame(columns=['Municipality_Name', 'Year', 'Metric', 'Value', 'Unit'])
tidy_remoteness['Municipality_Name'] = remoteness_subset['CSDname']
tidy_remoteness['Year'] = 2021
tidy_remoteness['Metric'] = 'Remoteness'
tidy_remoteness['Value'] = remoteness_subset['Index_of_remoteness']
tidy_remoteness['Unit'] = 'index'

tidy_remoteness.head()

,Municipality_Name,Year,Metric,Value,Unit
4304,Sparwood,2021,Remoteness,0.387226,index
4305,Fernie,2021,Remoteness,0.427617,index
4308,Cranbrook,2021,Remoteness,0.385609,index
4309,Kimberley,2021,Remoteness,0.409831,index
4312,Invermere,2021,Remoteness,0.460123,index


In [56]:
#save mathed dataframes to csv files in the data/processed folder
tidy_remoteness.to_csv(os.path.join('..','..', 'data', 'processed', 'tidy_remoteness.csv'), index=False)